# CS5491 — GNN+ normalization × depth sweep (Colab)

Runs the real ZINC grid on a **T4 GPU** and produces everything needed to fill
Table 3 of the final report.

**Before running, choose one way to get the code in:**
- **A. Public repo already exists** — set `REPO_URL` in cell 1. Recommended,
  because the final submission needs a public repo + commit hash anyway.
- **B. Not pushed yet** — leave `REPO_URL` empty and use the zip-upload cell 2.
  Push the repo afterwards, once the results have been copied back.

Runtime → Change runtime type → **T4 GPU**. The full grid is 5 norms × 4
depths × 3 seeds × 300 epochs = 60 training runs (several hours). For a smoke
check set `EPOCHS=20`, `REPEAT=1`, `NORMS='batch none'`, `DEPTHS='4 20'` in
cell 5. The sweep supports `--skip-existing`, so an interrupted run resumes.

Two preflight checks guard the long run: a ZINC download check and a
**one-cell 5-epoch training smoke test** that must write a
`test/stats.json` before the grid is allowed to start. Both stop the run
early with the actual traceback if anything is broken.

**Output:** `results/norm_study_summary.csv` (copy back into the repository's
`results/` folder) plus rebuilt milestone/final PDFs. The report generator
fills Table 3, redraws Figure 1 and writes the Section 4 conclusions
automatically from the CSV — no number is ever typed by hand.

In [ ]:
# Public repository for this project. It is already created and Public.
# IMPORTANT: push the local repository FIRST (step 1 of the handoff guide),
# otherwise this clone would be empty. If you have not pushed yet, set
# REPO_URL = '' below and use the zip-upload fallback in cell 2 instead.
REPO_URL = 'https://github.com/1589893076/CS5491-GNNPlus-NormStudy.git'
import os, subprocess

if REPO_URL and 'YOUR_NAME' not in REPO_URL:
    if not os.path.exists('main.py'):
        subprocess.run(['git', 'clone', REPO_URL, 'repo'], check=True)
        os.chdir('repo')
    assert os.path.exists('main.py'), (
        'Clone contains no code: push the local repository first, or set '
        "REPO_URL = '' and use the zip-upload cell below.")
    print('repo ready at', os.getcwd())
else:
    print('REPO_URL not set -> use the zip-upload cell below, then continue.')

In [ ]:
# Fallback B: upload the repository as a zip (e.g. GNNPlus-main.zip).
# Uncomment these three lines if you did not use a public repo, then re-run this cell.
# from google.colab import files
# files.upload()
# !unzip -q -o GNNPlus-main.zip -d /content/repo && %cd /content/repo
import os
assert os.path.exists('main.py'), (
    'Repository not found. Set REPO_URL in cell 1, or uncomment the zip upload above.')
print('working directory:', os.getcwd())

In [ ]:
import torch, os
from pathlib import Path

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'No GPU detected. Use Runtime → Change runtime type → T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

tv = torch.__version__.split('+')[0]
cuda_tag = ('cpu' if torch.version.cuda is None
            else 'cu' + torch.version.cuda.replace('.', ''))
wheel_index = f'https://data.pyg.org/whl/torch-{tv}+{cuda_tag}.html'
print('PyG wheel index:', wheel_index)

In [ ]:
!pip -q install torch_geometric scikit-learn fsspec rdkit pytorch-lightning yacs torchmetrics networkx tensorboardX ogb wandb reportlab pypdf
# Optional compiled extensions. If no wheel matches this runtime's torch,
# the pure-PyTorch fallback below keeps the sweep runnable.
!pip -q install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_index} || echo "compiled extension wheels unavailable -> fallback"

import os, shutil
from pathlib import Path
try:
    import torch_scatter  # noqa: F401
    print('torch_scatter: compiled extension available.')
except Exception as exc:
    root = Path(os.getcwd())
    shutil.copy(str(root / 'experiments' / 'torch_scatter_fallback.py'),
                str(root / 'torch_scatter.py'))
    os.environ['PYTHONPATH'] = (str(root) + os.pathsep
                                + os.environ.get('PYTHONPATH', ''))
    print('torch_scatter: using pure-PyTorch fallback because:', exc)

!python experiments/test_norm_smoke.py

# The smoke test above loads norm.py in isolation: it never imports GNNPlus.
# Training goes through main.py -> import GNNPlus -> every layer, and four of
# those modules import torch_scatter unconditionally. Verify the REAL import
# chain in a fresh subprocess (exactly how the sweep launches main.py), so a
# missing compiled extension cannot silently kill all 60 runs.
import subprocess, sys
_probe = subprocess.run(
    [sys.executable, '-c', 'import GNNPlus; print("GNNPlus import chain OK")'],
    cwd=os.getcwd(), capture_output=True, text=True)
print(_probe.stdout.strip())
if _probe.returncode:
    print(_probe.stderr[-2500:])
    raise RuntimeError(
        'GNNPlus import chain failed. Most likely torch_scatter has no wheel '
        'and the pure-PyTorch fallback is not on sys.path. Fix this before '
        'starting the grid.')

In [ ]:
# Preflight BEFORE the multi-hour sweep: confirm ZINC can actually download.
# GraphGym's default dataset dir is datasets/, and master_loader.py appends
# the format id 'ZINC', so this warms exactly the cache the training runs use.
# If this fails, do NOT start the grid: fix the network / upload the dataset
# first, otherwise all 60 runs would fail at the loader.
!python -c "from torch_geometric.datasets import ZINC; d = ZINC(root='datasets/ZINC', subset=True, split='train'); print('ZINC train graphs:', len(d), '| sample:', d[0])" 

In [ ]:
# One-cell end-to-end smoke test BEFORE the multi-hour grid.
# Proves that main.py can start, load ZINC, build the model, train and write
# test/stats.json. Import / dataset / CUDA errors show up here in minutes
# instead of after the whole sweep. --tag keeps this run in its own directory
# (the aggregator ignores tagged dirs, so it cannot pollute the real table).
!python experiments/run_norm_study.py --norms batch --depths 4 --repeat 1 --epochs 5 --tag smoke
from pathlib import Path
stats = Path('results/norm_study/zinc_norm_batch_d4_smoke/0/test/stats.json')
if not stats.exists() or not stats.read_text(encoding='utf-8').strip():
    log = Path('logs/norm_study/zinc_norm_batch_d4_smoke.log')
    print('--- last 60 lines of', log, '---')
    if log.exists():
        print('\n'.join(log.read_text(errors='replace').splitlines()[-60:]))
    raise RuntimeError(
        'One-cell smoke test produced no test/stats.json. Read the traceback '
        'above and fix it before running the full grid. Do NOT download an '
        'empty summary.')
print('One-cell smoke test OK:', stats, '| rows =',
      len(stats.read_text(encoding='utf-8').strip().splitlines()))

In [ ]:
# Full grid. Reduce EPOCHS / REPEAT / NORMS / DEPTHS for a quick smoke check.
EPOCHS = 300
REPEAT = 3
NORMS = 'batch layer graph pair none'
DEPTHS = '4 8 12 20'
!python experiments/run_norm_study.py --repeat {REPEAT} --epochs {EPOCHS} --norms {NORMS} --depths {DEPTHS} --skip-existing

In [ ]:
!python experiments/aggregate_norm_study.py
from IPython.display import display, Markdown
from pathlib import Path
import csv

summary = Path('results/norm_study_summary.csv')
if not summary.exists():
    raise RuntimeError(
        'No summary was written: the grid produced no stats.json. Go back to '
        'the sweep cell, read its error output, fix it and re-run before '
        'continuing. Do not download anything from this notebook yet.')
with summary.open(newline='', encoding='utf-8') as fh:
    rows = list(csv.DictReader(fh))
print('ZINC summary rows:', len(rows), '/ 20 planned cells')
if not rows:
    raise RuntimeError('Empty summary table: the sweep did not complete.')
if len(rows) < 20:
    print('WARNING: partial grid. The report will say so explicitly; '
          'consider re-running the missing cells with --skip-existing.')
md = Path('results/norm_study_summary.md')
if md.exists():
    display(Markdown(md.read_text(encoding='utf-8')))

In [ ]:
# Rebuild both PDFs into one folder (the generator otherwise writes into the
# course stage folders, which do not exist on Colab), then download everything.
import csv
from pathlib import Path
summary = Path('results/norm_study_summary.csv')
if not summary.exists():
    raise RuntimeError('No result CSV: do not download. Fix the sweep cell first.')
with summary.open(newline='', encoding='utf-8') as fh:
    n_rows = sum(1 for _ in csv.DictReader(fh))
print('result rows:', n_rows, '/ 20 planned cells')
if n_rows < 20:
    print('WARNING: partial grid. Reports will be labelled accordingly.')
!python docs/build_reports.py --out-dir /content/cs5491_reports
from google.colab import files
for f in [Path('results/norm_study_summary.csv'),
          Path('results/norm_study_summary.md'),
          Path('/content/cs5491_reports/CS5491_Final_Report.pdf'),
          Path('/content/cs5491_reports/CS5491_Milestone_Report.pdf')]:
    if f.exists():
        files.download(str(f))
        print('downloaded', f)
    else:
        print('MISSING — check the cell above:', f)

## Before you download

The one-cell smoke test must print `One-cell smoke test OK` first. If it
raises, read the traceback it prints and fix the problem before running
the grid; `--tag smoke` keeps that trial out of the real result table.

The aggregation cell must print `ZINC summary rows: 20 / 20 planned cells`.
If it prints fewer rows, or raises "No summary was written", the sweep cell
failed and produced no `stats.json`. Read that cell's error output, fix the
problem, re-run the grid (`--skip-existing` resumes completed cells), then
aggregate again. **Never copy an empty CSV back**: a header-only file cannot
fill Table 3, and the local finalization script will now reject it.

## After running

1. Copy `norm_study_summary.csv` (and `.md`) into
   `Final project/GNNPlus-main/results/` on your machine.
2. From the repository root run `python docs/finish_project.py`. It validates
   the grid, rebuilds `project milestone/CS5491_Milestone_Report.pdf` and
   `Final project/CS5491_Final_Report.pdf`, fills Table 3, redraws Figure 1
   and writes the Section 4 conclusions from the CSV.
3. Check the final PDF: still 4 pages, Table 3 contains no `pending`, and the
   numbers match the CSV.
4. Push the public repository, then submit its URL + commit hash on Canvas.
   **Do not push again after submitting.**